<a href="https://colab.research.google.com/github/saurabh202/Capstone_Project_code/blob/main/Capstone_code_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mount Google Drive to use Dataset

Dataset link : https://drive.google.com/drive/folders/1YC4D7eTf1_phvQvq9bbTL5YS_GcxL65C?usp=share_link


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

os.makedirs('/content/dataset', exist_ok=True)
!unzip -q "/content/drive/MyDrive/Capstone_Dataset/AAAI_dataset.zip" -d /content/dataset

# Verify
print('Dataset files:', os.listdir('/content/dataset/AAAI_dataset'))
print('Train images:', len(os.listdir('/content/dataset/AAAI_dataset/Images/gossip_train')))
print('Test images: ', len(os.listdir('/content/dataset/AAAI_dataset/Images/gossip_test')))

Dataset files: ['gossip_test.xlsx', 'gossip_test.csv', 'politi_train.csv', 'politi_test.csv', 'gossip_train.xlsx', 'Images', '.DS_Store', 'gossip_train.csv']
Train images: 9988
Test images:  2828


## Clone the fork from MIMoE-FND



In [ ]:
%cd /content
!git clone https://github.com/saurabh202/Capstone_Project_code.git MIMoE-FND
%cd /content/MIMoE-FND
!ls

/content
Cloning into 'MIMoE-FND'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 87 (delta 24), reused 40 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 2.52 MiB | 44.44 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/MIMoE-FND
Capstone_code_file.ipynb  models_mae.py  requirements.txt  util
data			  preprocessing  train_vimoe.py    visualizations
models			  README.md	 train_vimoe.sh


## Install Dependencies

In [ ]:
!pip install -q pytorch-warmup paramiko positional-encodings ftfy timm einops
!pip install -q git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.9/208.9 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## MAE Pretrained Download



Source: https://github.com/facebookresearch/mae

In [ ]:
%cd /content/MIMoE-FND
!wget -q --show-progress https://dl.fbaipublicfiles.com/mae/pretrain/mae_pretrain_vit_base.pth
!ls -lh mae_pretrain_vit_base.pth

/content/MIMoE-FND
mae_pretrain_vit_ba 100%[===================>] 327.35M   218MB/s    in 1.5s    
-rw-r--r-- 1 root root 328M Dec 29  2021 mae_pretrain_vit_base.pth


## Checkpoint for recording accuracies



In [ ]:
import os

os.makedirs('/content/drive/MyDrive/Capstone_Dataset/kg_data/checkpoints/baseline_gossip', exist_ok=True)

# Create the PARENT checkpoints/ folder, but not the gossip/ subfolder itself
!mkdir -p /content/MIMoE-FND/checkpoints
!rm -rf /content/MIMoE-FND/checkpoints/gossip   # just in case it exists as a real folder
!ln -s /content/drive/MyDrive/Capstone_Dataset/kg_data/checkpoints/baseline_gossip /content/MIMoE-FND/checkpoints/gossip

# Confirm it's a symlink (look for the arrow ->)
!ls -la /content/MIMoE-FND/checkpoints/

total 12
drwxr-xr-x 2 root root 4096 Jul 21 14:08 .
drwxr-xr-x 9 root root 4096 Jul 21 14:08 ..
lrwxrwxrwx 1 root root   75 Jul 21 14:08 gossip -> /content/drive/MyDrive/Capstone_Dataset/kg_data/checkpoints/baseline_gossip



##  Training batch_size=4 , epoch=1


In [7]:
%cd /content/MIMoE-FND

!python train_vimoe.py \
    -train_dataset gossip \
    -test_dataset  gossip \
    -batch_size    24     \
    -epochs        50     \
    -val           0      \
    -duplicate_fake_times 0 \
    -device        cuda:0 \
    -int_lr        1e-6   \
    -int_beta      0.1    \
    -agr_threshold 0.3    \
    -sem_threshold 0.3    \
    -note          "baseline_gossip"

/content/MIMoE-FND
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 224kB/s]
vocab.txt: 100% 110k/110k [00:00<00:00, 476kB/s]
tokenizer.json: 100% 269k/269k [00:00<00:00, 903kB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 274kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 793kB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 1.25MB/s]
preprocessor_config.json: 100% 316/316 [00:00<00:00, 2.07MB/s]
config.json: 100% 4.10k/4.10k [00:00<00:00, 13.4MB/s]
tokenizer_config.json: 100% 905/905 [00:00<00:00, 5.62MB/s]
vocab.json: 100% 961k/961k [00:00<00:00, 115MB/s]
merges.txt: 100% 525k/525k [00:00<00:00, 101MB/s]
tokenizer.json: 100% 2.22M/2.22M [00:00<00:00, 147MB/s]
special_tokens_map.json: 100% 389/389 [00:00<00:00, 2.74MB/s]
Namespace(output_file='./output', train_dataset='gossip', test_dataset='gossip', checkpoint='', device='cuda:0', finetune=0, val=0, duplicate_fake_times=0, batch_size=24, epochs=50, hidden_dim=512, embed_dim=32, vocab_size=25, text_only=False, get_MLP_score

## Best results for epoch 32

In [31]:
val_acc = 0.8856
img_acc = 0.7996389891696751
text_acc = 0.8855595667870036
real_precision = 0.9051833122629582
real_recall = 0.9589285714285715
real_acc = 0.9589285714285715
real_f1 = 0.9312811619336657
fake_precision = 0.7682619647355163
fake_recall = 0.5754716981132075
fake_acc = 0.5754716981132075
fake_f1 = 0.6580366774541533

print(f"Val_Acc: {val_acc:.3f}")
print("-------------------------------")
print(f"Image Accuracy: {img_acc:.3f}")
print(f"Text Accuracy: {text_acc:.3f}")
print()
print("-------Real News-------")
print()
print(f"Precision: {real_precision:.3f}")
print(f"Recall: {real_recall:.3f}")
print(f"Accuracy: {real_acc:.3f}")
print(f"F1: {real_f1:.3f}")
print()
print("-------Fake News-------")
print()
print(f"Precision: {fake_precision:.3f}")
print(f"Recall: {fake_recall:.3f}")
print(f"Accuracy: {fake_acc:.3f}")
print(f"F1: {fake_f1:.3f}")

epoch 32

Val_Acc: 0.886
-------------------------------
Image Accuracy: 0.800
Text Accuracy: 0.886

-------Real News-------

Precision: 0.905
Recall: 0.959
Accuracy: 0.959
F1: 0.931

-------Fake News-------

Precision: 0.768
Recall: 0.575
Accuracy: 0.575
F1: 0.658


## Best result for Fake News F1, epoch 33

In [35]:
fake_f1 = 0.6666666666666667
print("Fake News F1")
print()
print(f"F1: {fake_f1:.3f}")

Fake News F1

F1: 0.667


##KG implementation



In [ ]:
!pip install -q transformers

## Load data and to detect text column

In [ ]:
import pandas as pd

train_df = pd.read_excel('/content/dataset/AAAI_dataset/gossip_train.xlsx') # Train Dataset
test_df  = pd.read_excel('/content/dataset/AAAI_dataset/gossip_test.xlsx')  # Test Dataset

# Records Present in train and test split
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Columns:", train_df.columns.tolist())
train_df.head(2)

Train shape: (9759, 5)
Test shape: (2770, 5)
Columns: ['Unnamed: 0', 'content', 'image', 'label', 'type']


,Unnamed: 0,content,image,label,type
0,0,One Love Manchester: Katy Perry wears photos o...,DyweiEOWJPtOqslc31CyVnTCyvAm307J.jpg,1,multi
1,1,TLC cuts all ties with Derick Dillard after mo...,NSq6JnA5ziOJbAQCzVlfti3OymZvInGK.jpg,1,multi


In [ ]:
text_col = 'content' # column name in Gossipcop dataset
text_col

'content'

##  Load the NER pipeline

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

device = 0 if torch.cuda.is_available() else -1 # Use colabs GPU or if not then switch to CPU

# dslim BERT based model for NER
tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="max", # changed from "simple" to "max"
    device=device
)

print("NER pipeline loaded, using device:", "GPU" if device == 0 else "CPU")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NER pipeline loaded. Using device: GPU


## Entity Extraction on train and test





In [ ]:
from tqdm import tqdm #tqdm to show the progress bar

# func to run NER
def extract_entities(text):
    if not isinstance(text, str) or not text.strip(): # skip if not a valid text
        return []
    try:
        results = ner_pipeline(text[:512]) # fixed max input length 512 characters
        return [(r['word'], r['entity_group'], float(r['score'])) for r in results]
    except Exception as e:
        return []

tqdm.pandas()

train_df['entities'] = train_df['content'].progress_apply(extract_entities)
test_df['entities']  = test_df['content'].progress_apply(extract_entities)

print(train_df[['content', 'entities']].head(3))

100%|██████████| 2770/2770 [00:00<00:00, 454387.03it/s]

                                             content entities
0  One Love Manchester: Katy Perry wears photos o...       []
1  TLC cuts all ties with Derick Dillard after mo...       []
2  Cher Steals the Show at the 2017 Billboard Mus...       []


## Saving results of NER
saved as "entities" as column name and each row in style of word, entity_group, score

In [ ]:
import os, shutil

# Save locally in colab
os.makedirs('/content/MIMoE-FND/kg_data', exist_ok=True)
# Saved in pkl type files
train_df[['entities']].to_pickle('/content/MIMoE-FND/kg_data/train_entities.pkl')
test_df[['entities']].to_pickle('/content/MIMoE-FND/kg_data/test_entities.pkl')

# Save to drive
drive_path = '/content/drive/MyDrive/Capstone_Dataset/kg_data'
os.makedirs(drive_path, exist_ok=True)
shutil.copy('/content/MIMoE-FND/kg_data/train_entities.pkl', f'{drive_path}/train_entities.pkl')
shutil.copy('/content/MIMoE-FND/kg_data/test_entities.pkl', f'{drive_path}/test_entities.pkl')

print("Saved Drive:", os.listdir(drive_path))

KeyError: "None of [Index(['entities'], dtype='object')] are in the [columns]"

## Reload NER results from drive

In [ ]:
import pandas as pd

# Reload original dataset
train_df = pd.read_excel('/content/dataset/AAAI_dataset/gossip_train.xlsx')
test_df  = pd.read_excel('/content/dataset/AAAI_dataset/gossip_test.xlsx')

# Reload entities from Drive backup (no need to re-run NER)
train_entities = pd.read_pickle('/content/drive/MyDrive/Capstone_Dataset/kg_data/train_entities.pkl')
test_entities  = pd.read_pickle('/content/drive/MyDrive/Capstone_Dataset/kg_data/test_entities.pkl')

train_df['entities'] = train_entities['entities']
test_df['entities']  = test_entities['entities']

print("Restored. Train shape:", train_df.shape, "| Test shape:", test_df.shape)
print(train_df[['content', 'entities']].head(2))

Restored. Train shape: (9759, 6) | Test shape: (2770, 6)
                                             content  \
0  One Love Manchester: Katy Perry wears photos o...   
1  TLC cuts all ties with Derick Dillard after mo...   

                                            entities  
0  [(Manchester, MISC, 0.5110037326812744), (Katy...  
1  [(TLC, ORG, 0.9967092275619507), (Derick Dilla...  


##Prepare Entities for KG Linking

1. Collect all unique entity strings across train + test
2. Filter out junk (short characters, non-alphabetic characters, common english words)
3. Count how many articles each entity appears in

In [ ]:
import re # used for pattern matching
from collections import Counter

def is_valid_entity(text):
    text = text.strip() #Remove spaces in a text

    if len(text) < 2: # Short to classify meaninful entity
        return False
    if not re.search(r'[a-zA-Z]', text):  # r=raw string , # must contain at least one letter
        return False
    if text.lower() in ['the', 'a', 'an', 'and', 'of', 'in', 'on']:  # Reject common english words
        return False
    return True

# Collect all unique entity strings
all_entity_texts = set()  # set() removes duplicate
for ents in train_df['entities']:
    for e in ents:
        all_entity_texts.add(e[0])
for ents in test_df['entities']:
    for e in ents:
        all_entity_texts.add(e[0])

cleaned_entities = set(t.strip() for t in all_entity_texts if is_valid_entity(t))
print(f"Raw unique entities: {len(all_entity_texts)} -> Cleaned: {len(cleaned_entities)}")

Raw unique entities: 26183 -> Cleaned: 26077


In [ ]:
# Frequency count
entity_counter = Counter()

# For train
for ents in train_df['entities']:
    for e in ents:
        if e[0].strip() in cleaned_entities:
            entity_counter[e[0].strip()] += 1
# For Test
for ents in test_df['entities']:
    for e in ents:
        if e[0].strip() in cleaned_entities:
            entity_counter[e[0].strip()] += 1

print(f"Total unique entities: {len(entity_counter)}")
print(f"Entities appearing only once: {sum(1 for c in entity_counter.values() if c == 1)}")
print(f"Entities appearing >=2 times:  {sum(1 for c in entity_counter.values() if c >= 2)}")
print(f"Entities appearing >=5 times:  {sum(1 for c in entity_counter.values() if c >= 5)}")

print("\nTop 20 most frequent entities:")
for ent, count in entity_counter.most_common(20):
    print(f"  {ent}: {count}")

Total unique entities: 26077
Entities appearing only once: 15654
Entities appearing >=2 times:  10423
Entities appearing >=5 times:  3962

Top 20 most frequent entities:
  Meghan Markle: 639
  Harry: 622
  American: 594
  Kim Kardashian: 587
  Hollywood: 570
  Selena Gomez: 504
  Taylor Swift: 487
  Brad Pitt: 485
  Prince: 484
  Kylie Jenner: 483
  Instagram: 463
  Los Angeles: 420
  ABC: 375
  Justin Bieber: 370
  Angelina Jolie: 359
  Kanye West: 353
  Netflix: 322
  E! News: 320
  Jennifer Aniston: 313
  Kim: 306


##Entities that appear 2+ times are considered

In [ ]:
entities_to_link = [ent for ent, count in entity_counter.items() if count >= 2]
print(f"Entities to link: {len(entities_to_link)}")

Entities to link: 10423


##Wikidata Linking

API, User Agent, connection

In [ ]:
import requests

WIKIDATA_API = "https://www.wikidata.org/w/api.php" # Wikidata's public API
HEADERS = {
    "User-Agent": "Saurabh_CapstoneProject (research project)" # user-agent required as specified by API for source of request
}

session = requests.Session()
session.headers.update(HEADERS)

# To chek the connection
resp = session.get(WIKIDATA_API, params={
    "action": "wbsearchentities", "search": "Katy Perry",
    "language": "en", "format": "json", "limit": 1
}, timeout=5)
print("Status code:", resp.status_code)
print("Sample match:", resp.json()["search"][0]["label"], "-", resp.json()["search"][0]["description"])

Status code: 200
Sample match: Katy Perry - American singer


To read the pickle files (pkl) in binary

In [ ]:
import time, pickle, shutil, os
from tqdm import tqdm

# Two paths one for colab local and other on drive
CACHE_PATH = '/content/MIMoE-FND/kg_data/wikidata_cache.pkl'
DRIVE_CACHE_PATH = '/content/drive/MyDrive/Capstone_Dataset/kg_data/wikidata_cache.pkl'

# Load existing cache if present
try:
    with open(DRIVE_CACHE_PATH, 'rb') as f: # read binary
        entity_cache = pickle.load(f) # load a cache that may already exist from a previous run
    print(f"Loaded existing cache with {len(entity_cache)} entities.")
except FileNotFoundError:
    entity_cache = {} # No cache found
    print("No existing cache found, start again")

# To save cache
def save_cache():
    os.makedirs('/content/MIMoE-FND/kg_data', exist_ok=True)
    with open(CACHE_PATH, 'wb') as f: # wb=write binary
        pickle.dump(entity_cache, f)
    os.makedirs('/content/drive/MyDrive/Capstone_Dataset/kg_data', exist_ok=True)
    shutil.copy(CACHE_PATH, DRIVE_CACHE_PATH) # Copy local file to drive

Loaded existing cache with 10423 entities.


Failed vs Good

In [ ]:
# Clear any previously failed (None) entries to re run
before = len(entity_cache)
entity_cache = {k: v for k, v in entity_cache.items() if v is not None} # entries having value not None
print(f"Cleared {before - len(entity_cache)} failed entries. Remaining good: {len(entity_cache)}")

Cleared 1160 failed entries. Remaining good: 9263


In [ ]:
remaining = [e for e in entities_to_link if e not in entity_cache] # Remaining meaning subtracting the good ones
print(f"Remaining to link: {len(remaining)}")

consecutive_429s = 0

for i, entity_text in enumerate(tqdm(remaining)):
    success = False
    # Trying 3 times per entity before failing
    for attempt in range(3):
        try:
            resp = session.get(WIKIDATA_API, params={
                "action": "wbsearchentities", "search": entity_text,
                "language": "en", "format": "json", "limit": 1
            }, timeout=10)

            if resp.status_code == 429: # 429 = too many request
                consecutive_429s += 1
                wait = int(resp.headers.get("Retry-After", 5)) # 5 seconds to wait
                time.sleep(wait)
                continue

            consecutive_429s = 0
            data = resp.json()

            # If matched save the top result's ID, label, and description
            if data.get("search"):
                top = data["search"][0]
                entity_cache[entity_text] = {
                    "qid": top["id"],
                    "label": top.get("label", entity_text),
                    "description": top.get("description", "")
                }
            else:
                entity_cache[entity_text] = None # No matching entity
            success = True
            break
        except Exception:
            time.sleep(2) # If exception occured wait 2 sec and try agian

    if not success:
        entity_cache[entity_text] = None

    time.sleep(1.0)  # 1 request per sec for wikidata

    # If >=5 errors then stop
    if consecutive_429s >= 5:
        print("Too many request, fail, Stop")
        save_cache()
        break

    # To save results after evry 200 entities to drive
    if i % 200 == 0 and i > 0:
        save_cache()

save_cache() # Save when completes
print(f"Done for now. Cache has {len(entity_cache)} entities.")
good = sum(1 for v in entity_cache.values() if v is not None)
print(f"Success rate: {good/len(entity_cache)*100:.1f}%")

Remaining to link: 1160


  0%|          | 3/1160 [00:03<24:30,  1.27s/it]


KeyboardInterrupt: 

##Entities successfully linked vs failed

In [ ]:
good = sum(1 for v in entity_cache.values() if v is not None) # success not None
bad = sum(1 for v in entity_cache.values() if v is None) # Failed has None
print(f"Successfully linked: {good}")
print(f"Failed / no match: {bad}")

# Randomly select 10 entities
import random
sample_keys = random.sample([k for k,v in entity_cache.items() if v is not None], 10)
print("\nSample linked entities:")
for k in sample_keys:
    print(f"  {k} -> {entity_cache[k]['qid']} | {entity_cache[k]['label']} | {entity_cache[k]['description']}")

Successfully linked: 9263
Failed / no match: 0

Sample linked entities:
  Lucas -> Q12325000 | Lucas | male given name
  Aussie -> Q408 | Australia | country in Oceania
  HIV -> Q15787 | HIV | human retrovirus, cause of AIDS
  Los Angeles Hospital -> Q17654268 | Los Angeles hospital lies, discriminates for Saudi liver transplant patient | Wikinews article
  Ross Mathews -> Q7369503 | Ross Mathews | American television personality
  Sterling K Brown -> Q3295000 | Sterling K. Brown | American actor
  Road Rules -> Q3074 | traffic code | traffic regulations, laws, and rules
  Fiji -> Q712 | Fiji | island sovereign state in Oceania
  Egypt -> Q79 | Egypt | country in Northeast Africa and Southwest Asia
  Maddox Jolie -> Q13147738 | Maddox Jolie | actor and producer


In [ ]:
generic_terms = ['male given name', 'family name']

generic_count = sum(
    1 for v in entity_cache.values()
    if v is not None and any(term in v['description'].lower() for term in generic_terms)
)

print(f"Entities with generic descriptions: {generic_count}")

Entities with generic descriptions: 1625


## Spacy Implementation


In [ ]:
!pip install -q spacy-entity-linker
!python -m spacy download en_core_web_md -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 27.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
import spacy_entity_linker  # registers the entityLinker component with spaCy
import re

nlp = spacy.load("en_core_web_md")
nlp.add_pipe("entityLinker", last=True)

# Use spaCy's full standard stopword list (~326 words) instead of a hand-typed short list
STOPWORDS = nlp.Defaults.stop_words
print("Stop words removed")

Stop words removed


## Entity Cleaning + Final Context-Aware Linking Function

In [ ]:
def clean_entity_text(text):
    """Strip stray quote marks and punctuation from entity text edges."""
    text = re.sub(r'^[\s"\'"''""„‚]+|[\s"\'"''""„‚]+$', '', text)
    return text.strip()

def is_valid_entity(text):
    """Filter out junk: too short, no letters, or a common stopword."""
    text = clean_entity_text(text)
    if len(text) < 2:
        return False
    if not re.search(r'[a-zA-Z]', text):
        return False
    if text.lower() in STOPWORDS:
        return False
    return True

def link_article_entities(article_text, ner_entities_with_labels, wikidata_cache):
    """
    Entity resolution per article:
    1. Clean and deduplicate BERT NER's extracted entities
    2. Try spaCy's context-aware disambiguation
    3. Block spaCy's answer if BERT says PERSON but the linked entity's own
       description indicates a place (prevents wrong-type matches like "Dillard"->city)
    4. Fall back to existing Wikidata cache
    5. Drop entities neither source can resolve
    """
    raw_texts = list(set(
        clean_entity_text(e[0]) for e in ner_entities_with_labels
        if is_valid_entity(e[0])
    ))
    label_lookup = {clean_entity_text(e[0]): e[1] for e in ner_entities_with_labels}

    doc = nlp(article_text[:1000])
    context_links = {}
    for ent in doc._.linkedEntities:
        key = str(ent.get_span()).lower().strip()
        context_links[key] = {
            "qid": f"Q{ent.get_id()}",
            "label": ent.get_label(),
            "description": ent.get_description()
        }

    PERSON = {'PER', 'PERSON'}
    LOCATION_WORDS = ['city', 'town', 'village', 'county', 'municipality', 'commune',
                       'country', 'state', 'province', 'district']

    final = {}
    for ent_text in raw_texts:
        key = ent_text.lower()
        ner_label = label_lookup.get(ent_text, 'MISC')

        if key in context_links:
            desc = context_links[key]['description'].lower()
            is_location_match = any(w in desc for w in LOCATION_WORDS)

            if ner_label in PERSON and is_location_match:
                pass
            else:
                final[ent_text] = {**context_links[key], "source": "spacy_contextual"}
                continue

        if ent_text in wikidata_cache and wikidata_cache[ent_text] is not None:
            final[ent_text] = {**wikidata_cache[ent_text], "source": "wikidata_cache"}

    return final

<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1176/1045314811.py:3: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(r'^[\s"\'"''""„‚]+|[\s"\'"''""„‚]+$', '', text)


## Load Final Wikidata Cache





In [ ]:
import pickle

with open('/content/drive/MyDrive/Capstone_Dataset/kg_data/final_linked_entities.pkl', 'rb') as f:
    final_entities = pickle.load(f)

print(f"Loaded {len(final_entities)} entities from final_linked_entities.pkl")

Loaded 7472 entities from final_linked_entities.pkl


## Clear Previous Results (before regenerating)

In [ ]:
import os

article_entities = {}
for path in ['/content/MIMoE-FND/kg_data/final_article_entities.pkl',
             '/content/drive/MyDrive/Capstone_Dataset/kg_data/final_article_entities.pkl']:
    if os.path.exists(path):
        os.remove(path)
print("Cleared. Ready to regenerate.")

Cleared. Ready to regenerate.


## Run Full Pipeline Across All Articles

In [ ]:
from tqdm import tqdm
import pickle, shutil, os

FINAL_CACHE_PATH = '/content/MIMoE-FND/kg_data/final_article_entities.pkl'
DRIVE_FINAL_PATH = '/content/drive/MyDrive/Capstone_Dataset/kg_data/final_article_entities.pkl'

try:
    with open(DRIVE_FINAL_PATH, 'rb') as f:
        article_entities = pickle.load(f)
    print(f"Resumed: {len(article_entities)} articles already done")
except FileNotFoundError:
    article_entities = {}
    print("Starting fresh")

def save_progress():
    os.makedirs('/content/MIMoE-FND/kg_data', exist_ok=True)
    with open(FINAL_CACHE_PATH, 'wb') as f:
        pickle.dump(article_entities, f)
    os.makedirs('/content/drive/MyDrive/Capstone_Dataset/kg_data', exist_ok=True)
    shutil.copy(FINAL_CACHE_PATH, DRIVE_FINAL_PATH)

all_articles = [(f"train_{i}", row['content'], row['entities']) for i, row in train_df.iterrows()]
all_articles += [(f"test_{i}", row['content'], row['entities']) for i, row in test_df.iterrows()]

remaining = [a for a in all_articles if a[0] not in article_entities]
print(f"Remaining articles to process: {len(remaining)}")

for i, (aid, text, ents) in enumerate(tqdm(remaining)):
    if not isinstance(text, str) or not text.strip():
        article_entities[aid] = {}
        continue
    try:
        article_entities[aid] = link_article_entities(text, ents, final_entities)
    except Exception:
        article_entities[aid] = {}
    if i % 500 == 0 and i > 0:
        save_progress()

save_progress()
print(f"Done. {len(article_entities)} articles processed.")

Starting fresh
Remaining articles to process: 12529


100%|██████████| 12529/12529 [11:02<00:00, 18.91it/s]


Done. 12529 articles processed.


In [ ]:
sample_ids = ["train_0", "train_1", "train_2"]

In [ ]:
print("AFTER — Final Refined, Disambiguated, Wikidata-Linked Entities\n")
for aid in sample_ids:
    idx = int(aid.split("_")[1])
    row = train_df.loc[idx]
    print(f"Article: {row['content'][:90]}...")
    for text, data in article_entities[aid].items():
        print(f"  {text} -> QID: {data['qid']} | {data['label']} | {data['description']} | source: {data['source']}")
    print()

AFTER — Final Refined, Disambiguated, Wikidata-Linked Entities

Article: One Love Manchester: Katy Perry wears photos of 22 victims She was sending a thought-provo...


NameError: name 'article_entities' is not defined

## Building the Knowledge Graph


## Relationship Properties

##

In [ ]:
!pip install -q pykeen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 46.3 MB/s eta 0:00:00


##Wikidata Relationship Fetching Functions

In [ ]:
import time

def fetch_entity_relations_batch(qids):
    params = {
        "action": "wbgetentities",
        "ids": "|".join(qids),
        "props": "claims",
        "format": "json"
    }
    try:
        resp = session.get(WIKIDATA_API, params=params, timeout=15)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            time.sleep(wait)
            return None
        return resp.json()
    except Exception:
        return None

def extract_all_entity_relations(claims):
    relations = []
    for prop_id, claim_list in claims.items():
        for claim in claim_list:
            try:
                datavalue = claim["mainsnak"]["datavalue"]
                if datavalue["type"] == "wikibase-entityid":
                    target_qid = datavalue["value"]["id"]
                    relations.append((prop_id, target_qid))
            except (KeyError, TypeError):
                continue
    return relations

In [ ]:
all_qids = set()
for ents in article_entities.values():
    for data in ents.values():
        all_qids.add(data['qid'])
all_qids = list(all_qids)
print(f"Total unique entities (QIDs): {len(all_qids)}")

NameError: name 'article_entities' is not defined

In [ ]:
print(f"Current entity_relations count: {len(entity_relations)}")

NameError: name 'entity_relations' is not defined

## Relationship Fetching
knowledge graph structure

In [ ]:
from tqdm import tqdm
import pickle, shutil, os

RELATIONS_CACHE_PATH = '/content/MIMoE-FND/kg_data/entity_relations.pkl'
DRIVE_RELATIONS_PATH = '/content/drive/MyDrive/Capstone_Dataset/kg_data/entity_relations.pkl'

def save_relations_cache():
    os.makedirs('/content/MIMoE-FND/kg_data', exist_ok=True)
    with open(RELATIONS_CACHE_PATH, 'wb') as f:
        pickle.dump(entity_relations, f)
    os.makedirs('/content/drive/MyDrive/Capstone_Dataset/kg_data', exist_ok=True)
    shutil.copy(RELATIONS_CACHE_PATH, DRIVE_RELATIONS_PATH)

remaining_qids = [q for q in all_qids if q not in entity_relations]
print(f"Remaining QIDs to fetch: {len(remaining_qids)}")

batch_size = 50
batches = [remaining_qids[i:i+batch_size] for i in range(0, len(remaining_qids), batch_size)]

for i, batch in enumerate(tqdm(batches)):
    data = fetch_entity_relations_batch(batch)
    if data is None:
        continue
    entities_data = data.get("entities", {})
    for qid, edata in entities_data.items():
        claims = edata.get("claims", {})
        entity_relations[qid] = extract_all_entity_relations(claims)
    time.sleep(1.0)
    if i % 20 == 0 and i > 0:
        save_relations_cache()

save_relations_cache()
print(f"Done. Fetched relations for {len(entity_relations)} entities.")

Remaining QIDs to fetch: 10469


100%|██████████| 210/210 [18:49<00:00,  5.38s/it]


Done. Fetched relations for 12319 entities.


## Reload saved progress

In [ ]:
import pickle

with open('/content/drive/MyDrive/Capstone_Dataset/kg_data/entity_relations.pkl', 'rb') as f:
    entity_relations = pickle.load(f)

print(f"Reloaded: {len(entity_relations)} entities")

Reloaded: 12319 entities


## Total batches 266

In [ ]:
print(f"Final count: {len(entity_relations)}")

Final count: 12319


##  PyKEEN training

## Build Knowledge Graph Triples from Fetched Relations

In [ ]:
import pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline

triples = []
for head_qid, relations in entity_relations.items():
    for prop_id, tail_qid in relations:
        triples.append((head_qid, prop_id, tail_qid))

triples_df = pd.DataFrame(triples, columns=['head', 'relation', 'tail'])
print(f"Total triples: {len(triples_df)}")

INFO:pykeen.utils:Using opt_einsum


Total triples: 383380


## Split into Train / Validation / Test

In [ ]:
tf = TriplesFactory.from_labeled_triples(triples_df.values)
print(f"Unique entities: {tf.num_entities}")
print(f"Unique relation types: {tf.num_relations}")

training, validation, testing = tf.split([0.8, 0.1, 0.1], random_state=42)
print(f"Training: {training.num_triples}, Validation: {validation.num_triples}, Testing: {testing.num_triples}")

Unique entities: 165101
Unique relation types: 989


INFO:pykeen.triples.splitting:done splitting triples to groups of sizes [141426, 37663, 37663]


Training: 301300, Validation: 37663, Testing: 37663


##

In [ ]:
import os

checkpoint_path = '/content/drive/MyDrive/Capstone_Dataset/kg_data/checkpoints/transe_checkpoint.pt'
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print("Removed old checkpoint")
else:
    print("No old checkpoint found")

Removed old checkpoint


## Train Knowledge Graph Embeddings (TransE via PyKEEN)

In [ ]:
result = pipeline(
    training=training,
    validation=validation,
    testing=testing,
    model="TransE",
    model_kwargs=dict(embedding_dim=128),
    training_kwargs=dict(
        num_epochs=100,
        checkpoint_name='transe_checkpoint.pt',
        checkpoint_directory='/content/drive/MyDrive/Capstone_Dataset/kg_data/checkpoints',
        checkpoint_frequency=5,
        checkpoint_on_failure=True,
    ),
    stopper="early",
    stopper_kwargs=dict(frequency=5, patience=3, relative_delta=0.01),
    random_seed=42
)
print("Training complete.")

INFO:pykeen.pipeline.api:=> no training loop checkpoint file found at '/content/drive/MyDrive/Capstone_Dataset/kg_data/checkpoints/transe_checkpoint.pt'. Creating a new file.
INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.stoppers.early_stopping:Inferred checkpoint path for best model weights: /root/.data/pykeen/checkpoints/best-model-weights-9dc863ba-16e7-409c-af5b-58855a7be228.pt
INFO:pykeen.training.training_loop:=> no checkpoint found at '/content/drive/MyDrive/Capstone_Dataset/kg_data/checkpoints/transe_checkpoint.pt'. Creating a new file.


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/1.18k [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/1.18k [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/1.18k [00:00<?, ?batch/s]

...

Training batches on cuda:0:   0%|          | 0.00/1.18k [00:00<?, ?batch/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 92.14s seconds
INFO:pykeen.stoppers.early_stopping:Stopping early at epoch 40. The best result 0.1507314871359159 occurred at epoch 25.
INFO:pykeen.stoppers.early_stopping:Re-loading weights from best epoch from /root/.data/pykeen/checkpoints/best-model-weights-9dc863ba-16e7-409c-af5b-58855a7be228.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 40.


Evaluating on cuda:0:   0%|          | 0.00/37.7k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 91.95s seconds


Training complete.


In [ ]:
# Full metrics dict (MRR, MR, Hits@1/3/5/10, etc.)
print(pipeline_result.metric_results.to_df())

# Just the headline numbers
metrics = pipeline_result.metric_results.to_dict()
print(metrics['both']['realistic']['inverse_harmonic_mean_rank'])  # MRR
print(metrics['both']['realistic']['hits_at_10'])
print(metrics['both']['realistic']['hits_at_3'])
print(metrics['both']['realistic']['hits_at_1'])

NameError: name 'pipeline_result' is not defined

In [ ]:
print(result.metric_results.to_df())

NameError: name 'result' is not defined

In [ ]:
for name, val in list(globals().items()):
    if 'pipeline' in name.lower() or hasattr(val, 'metric_results'):
        print(name, type(val))

pipeline <class 'function'>


In [ ]:
print([v for v in dir() if not v.startswith('_')])

['CACHE_PATH', 'CHECKPOINT_DIR', 'Counter', 'DRIVE_BASE', 'DRIVE_CACHE_PATH', 'HEADERS', 'In', 'Out', 'RESULTS_DIR', 'STOPWORDS', 'TriplesFactory', 'WIKIDATA_API', 'aid', 'all_entity_texts', 'all_qids', 'attempt', 'bad', 'before', 'checkpoint_path', 'clean_entity_text', 'cleaned_entities', 'consecutive_429s', 'count', 'data', 'drive', 'e', 'ent', 'entities_to_link', 'entity_cache', 'entity_counter', 'entity_relations', 'entity_text', 'ents', 'exit', 'extract_all_entity_relations', 'f', 'fetch_entity_relations_batch', 'final_entities', 'generic_count', 'generic_terms', 'get_ipython', 'good', 'head_qid', 'i', 'idx', 'is_valid_entity', 'k', 'link_article_entities', 'name', 'nlp', 'os', 'pd', 'pickle', 'pipeline', 'prop_id', 'quit', 'random', 're', 'relations', 'remaining', 'requests', 'resp', 'rotate_checkpoint_name', 'row', 'sample_ids', 'sample_keys', 'save_cache', 'session', 'shutil', 'spacy', 'spacy_entity_linker', 'success', 'tail_qid', 'test_df', 'test_entities', 'testing', 'text_co

In [ ]:
print(result.metric_results.to_df())

NameError: name 'result' is not defined

In [ ]:
df = result.metric_results.to_df()

# Filter to the metrics people actually report
key_metrics = ['inverse_harmonic_mean_rank', 'hits_at_1', 'hits_at_3', 'hits_at_5', 'hits_at_10']

summary = df[
    (df['Metric'].isin(key_metrics)) &
    (df['Rank_type'] == 'realistic') &
    (df['Side'] == 'both')
]
print(summary)

NameError: name 'result' is not defined

In [ ]:
print(f"Entities: {training.num_entities}")
print(f"Relations: {training.num_relations}")
print(f"Train triples: {training.num_triples}")
print(f"Validation triples: {validation.num_triples}")
print(f"Test triples: {testing.num_triples}")
print(f"Triples per entity (avg): {training.num_triples / training.num_entities:.2f}")

NameError: name 'training' is not defined

## RoattE

In [ ]:
import os

DRIVE_BASE = '/content/drive/MyDrive/Capstone_Dataset/kg_data'
CHECKPOINT_DIR = f'{DRIVE_BASE}/checkpoints'
RESULTS_DIR = f'{DRIVE_BASE}/results/rotate'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

rotate_checkpoint_name = 'rotate_checkpoint.pt'

## Train RotatE

In [ ]:
!pip install pykeen -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 42.7 MB/s eta 0:00:00


In [ ]:
import os

base_path = '/content/drive/MyDrive/Capstone_Dataset/kg_data'
checkpoint_dir = f'{base_path}/checkpoints'
output_dir = f'{base_path}/model2_output'

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

checkpoint_file = 'model2_checkpoint.pt'

In [ ]:
from pykeen.triples import TriplesFactory

training = TriplesFactory.from_path_binary(f'{base_path}/training_tf')
validation = TriplesFactory.from_path_binary(f'{base_path}/validation_tf')
testing = TriplesFactory.from_path_binary(f'{base_path}/testing_tf')

print("Entities:", training.num_entities)
print("Relations:", training.num_relations)
print("Train triples:", training.num_triples)

INFO:pykeen.triples.triples_factory:Loading from file:///content/drive/MyDrive/Capstone_Dataset/kg_data/training_tf


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Capstone_Dataset/kg_data/training_tf/base.pth'

In [ ]:
import os
for f in os.listdir(base_path):
    print(f)

model2_output
results
checkpoints


In [ ]:
# Pattern A - single file split automatically
tf = TriplesFactory.from_path('some_triples_file.tsv')
training, validation, testing = tf.split([0.8, 0.1, 0.1])

FileNotFoundError: [Errno 2] No such file or directory: '/content/some_triples_file.tsv'

In [ ]:
# Pattern B - already split into 3 files
training = TriplesFactory.from_path('train.tsv')
validation = TriplesFactory.from_path('validation.tsv', entity_to_id=training.entity_to_id, relation_to_id=training.relation_to_id)
testing = TriplesFactory.from_path('test.tsv', entity_to_id=training.entity_to_id, relation_to_id=training.relation_to_id)

FileNotFoundError: [Errno 2] No such file or directory: '/content/train.tsv'

In [ ]:
# Pattern C - built from a pandas dataframe
tf = TriplesFactory.from_labeled_triples(triples_array)
training, validation, testing = tf.split(...)

NameError: name 'triples_array' is not defined

In [ ]:
import os

search_root = '/content/drive/MyDrive/Capstone_Dataset'
for dirpath, dirnames, filenames in os.walk(search_root):
    for f in filenames:
        if f.endswith(('.csv', '.tsv', '.pkl', '.parquet', '.txt')):
            full_path = os.path.join(dirpath, f)
            size_mb = os.path.getsize(full_path) / (1024*1024)
            print(f"{full_path}  ({size_mb:.1f} MB)")

In [ ]:
try:
    print(training.num_triples)
except NameError:
    print("Not in memory — needs rebuild")

Not in memory — needs rebuild


In [ ]:
import os

search_root = '/content/drive/MyDrive/Capstone_Dataset'
for dirpath, dirnames, filenames in os.walk(search_root):
    for f in filenames:
        if f.endswith(('.csv', '.tsv', '.pkl', '.parquet', '.txt', '.json')):
            full_path = os.path.join(dirpath, f)
            size_mb = os.path.getsize(full_path) / (1024*1024)
            print(f"{full_path}  ({size_mb:.1f} MB)")

In [ ]:
import os
print(os.path.exists('/content/drive/MyDrive/Capstone_Dataset'))
print(os.listdir('/content/drive/MyDrive/Capstone_Dataset'))

True
['kg_data']


In [ ]:
base_path = '/content/drive/MyDrive/Capstone_Dataset/kg_data'
for dirpath, dirnames, filenames in os.walk(base_path):
    print("DIR:", dirpath)
    for f in filenames:
        print("   file:", f)

DIR: /content/drive/MyDrive/Capstone_Dataset/kg_data
DIR: /content/drive/MyDrive/Capstone_Dataset/kg_data/model2_output
DIR: /content/drive/MyDrive/Capstone_Dataset/kg_data/results
DIR: /content/drive/MyDrive/Capstone_Dataset/kg_data/results/rotate
DIR: /content/drive/MyDrive/Capstone_Dataset/kg_data/checkpoints


In [ ]:
for dirpath, dirnames, filenames in os.walk('/content/drive/MyDrive/Capstone_Dataset'):
    print("DIR:", dirpath)
    for f in filenames:
        print("   file:", f)

DIR: /content/drive/MyDrive/Capstone_Dataset
DIR: /content/drive/MyDrive/Capstone_Dataset/kg_data
DIR: /content/drive/MyDrive/Capstone_Dataset/kg_data/model2_output
DIR: /content/drive/MyDrive/Capstone_Dataset/kg_data/results
DIR: /content/drive/MyDrive/Capstone_Dataset/kg_data/results/rotate
DIR: /content/drive/MyDrive/Capstone_Dataset/kg_data/checkpoints


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ValueError: Mountpoint must not already contain files

In [ ]:
try:
    print("Ready. Entities:", training.num_entities, "| Triples:", training.num_triples)
except NameError:
    print("Not ready yet — training/validation/testing still missing")

Ready. Entities: 165101 | Triples: 301300


## saving

In [ ]:
training.to_path_binary('/content/drive/MyDrive/Capstone_Dataset/kg_data/training_tf')
validation.to_path_binary('/content/drive/MyDrive/Capstone_Dataset/kg_data/validation_tf')
testing.to_path_binary('/content/drive/MyDrive/Capstone_Dataset/kg_data/testing_tf')
print("Saved.")

INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=165101, num_relations=989, create_inverse_triples=False, num_triples=301300) to file:///content/drive/MyDrive/Capstone_Dataset/kg_data/training_tf
INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=165101, num_relations=989, create_inverse_triples=False, num_triples=37663) to file:///content/drive/MyDrive/Capstone_Dataset/kg_data/validation_tf
INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=165101, num_relations=989, create_inverse_triples=False, num_triples=37663) to file:///content/drive/MyDrive/Capstone_Dataset/kg_data/testing_tf


Saved.


## RotatE

In [ ]:
import os
from pykeen.pipeline import pipeline

base_path = '/content/drive/MyDrive/Capstone_Dataset/kg_data'
checkpoint_dir = f'{base_path}/checkpoints'
output_dir = f'{base_path}/model2_output'
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
checkpoint_file = 'model2_checkpoint.pt'

result2 = pipeline(
    training=training,
    validation=validation,
    testing=testing,
    model='RotatE',
    model_kwargs=dict(embedding_dim=256),
    loss='NSSALoss',
    negative_sampler_kwargs=dict(num_negs_per_pos=20),
    training_kwargs=dict(
        num_epochs=150,
        batch_size=1024,
        checkpoint_name=checkpoint_file,
        checkpoint_directory=checkpoint_dir,
        checkpoint_frequency=5,
    ),
    stopper='early',
    stopper_kwargs=dict(patience=6, frequency=5, relative_delta=0.002),
    random_seed=42,
)
print("Training complete.")

INFO:pykeen.pipeline.api:=> no training loop checkpoint file found at '/content/drive/MyDrive/Capstone_Dataset/kg_data/checkpoints/model2_checkpoint.pt'. Creating a new file.
INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.stoppers.early_stopping:Inferred checkpoint path for best model weights: /root/.data/pykeen/checkpoints/best-model-weights-a53329c9-901e-4899-9b9a-0f94b39e03bc.pt
INFO:pykeen.training.training_loop:=> no checkpoint found at '/content/drive/MyDrive/Capstone_Dataset/kg_data/checkpoints/model2_checkpoint.pt'. Creating a new file.


Training epochs on cuda:0:   0%|          | 0/150 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/295 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/295 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/295 [00:00<?, ?batch/s]

...

Training batches on cuda:0:   0%|          | 0.00/295 [00:00<?, ?batch/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 310.73s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 150: 0.39651647505509385. Saved model weights to /root/.data/pykeen/checkpoints/best-model-weights-a53329c9-901e-4899-9b9a-0f94b39e03bc.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 150.
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 150.


Evaluating on cuda:0:   0%|          | 0.00/37.7k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 314.59s seconds


Training complete.


In [ ]:
import json, torch

output_dir = '/content/drive/MyDrive/Capstone_Dataset/kg_data/model2_output'

# 1. Full metrics table
metrics_df = result2.metric_results.to_df()
metrics_df.to_csv(f'{output_dir}/metrics.csv', index=False)

# 2. Summary (MRR, Hits@k)
summary = metrics_df.query(
    "Side=='both' and Rank_type=='realistic' and Metric in "
    "['inverse_harmonic_mean_rank','hits_at_1','hits_at_3','hits_at_5','hits_at_10']"
).set_index('Metric')['Value'].to_dict()

with open(f'{output_dir}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

# 3. Entity + relation embeddings
entity_embeddings = result2.model.entity_representations[0](indices=None).detach().cpu()
relation_embeddings = result2.model.relation_representations[0](indices=None).detach().cpu()

torch.save({
    'entity_embeddings': entity_embeddings,
    'relation_embeddings': relation_embeddings,
    'entity_to_id': training.entity_to_id,
    'relation_to_id': training.relation_to_id,
}, f'{output_dir}/embeddings.pt')

# 4. Full model object
torch.save(result2.model, f'{output_dir}/full_model.pt')

print("Saved everything to:", output_dir)
print("Entity embeddings shape:", entity_embeddings.shape)
print("Relation embeddings shape:", relation_embeddings.shape)

{
  "inverse_harmonic_mean_rank": 0.27490532398223877,
  "hits_at_1": 0.19847064758516317,
  "hits_at_3": 0.3034277673047819,
  "hits_at_5": 0.35342378461620155,
  "hits_at_10": 0.4206648434803388
}
Saved everything to: /content/drive/MyDrive/Capstone_Dataset/kg_data/model2_output
Entity embeddings shape: torch.Size([165101, 256])
Relation embeddings shape: torch.Size([989, 256])


## Reload the results of RotatE

In [ ]:
import torch, json

output_dir = '/content/drive/MyDrive/Capstone_Dataset/kg_data/model2_output'

kge_data = torch.load(f'{output_dir}/embeddings.pt')
entity_embeddings = kge_data['entity_embeddings']
relation_embeddings = kge_data['relation_embeddings']

with open(f'{output_dir}/summary.json') as f:
    summary = json.load(f)
print(summary)

# only if you need to keep using the model object itself:
model = torch.load(f'{output_dir}/full_model.pt', map_location='cuda:0',weights_only=False)

{'inverse_harmonic_mean_rank': 0.27490532398223877, 'hits_at_1': 0.19847064758516317, 'hits_at_3': 0.3034277673047819, 'hits_at_5': 0.35342378461620155, 'hits_at_10': 0.4206648434803388}


INFO:pykeen.utils:Using opt_einsum


In [ ]:
print(train_df['label'].value_counts())
print(train_df['label'].value_counts(normalize=True))

label
1    7746
0    2013
Name: count, dtype: int64
label
1    0.793729
0    0.206271
Name: proportion, dtype: float64


In [ ]:
try:
    print("Exists. Sample:", list(article_entities.items())[:1])
except NameError:
    print("Not in memory — needs rebuild")

Not in memory — needs rebuild


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle, torch, os

BASE = '/content/drive/MyDrive/Capstone_Dataset/kg_data'

# Linked entities per article (Step 2 output)
with open(f'{BASE}/final_article_entities.pkl', 'rb') as f:
    article_entities = pickle.load(f)

# Wikidata cache -- entity text -> {qid, description, ...} (Step 2)
with open(f'{BASE}/wikidata_cache.pkl', 'rb') as f:
    entity_cache = pickle.load(f)

# Fetched relationships -- qid -> [(prop_id, tail_qid), ...] (Step 3)
with open(f'{BASE}/entity_relations.pkl', 'rb') as f:
    entity_relations = pickle.load(f)

# Trained RotatE embeddings (Step 3, best-performing model)
kge_data = torch.load(f'{BASE}/model2_output/embeddings.pt')
entity_embeddings = kge_data['entity_embeddings']
entity_to_id = kge_data['entity_to_id']

print(f"Articles: {len(article_entities)}")
print(f"Relation-linked entities: {len(entity_relations)}")
print(f"RotatE embeddings: {entity_embeddings.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Articles: 12529
Relation-linked entities: 12319
RotatE embeddings: torch.Size([165101, 256])


## Per-Article Graphs

In [ ]:
!pip install -q torch_geometric

In [ ]:
from torch_geometric.data import Data

KG_DIM = entity_embeddings.shape[1]

def build_article_graph(entities_dict):
    qids, feats = [], []
    for _, data in entities_dict.items():
        qid = data.get('qid')
        if qid and qid in entity_to_id:
            qids.append(qid)
            feats.append(entity_embeddings[entity_to_id[qid]])

    if not qids:
        return None

    idx = {q: i for i, q in enumerate(qids)}
    x = torch.stack(feats)

    edges = []
    local_set = set(qids)
    for q in qids:
        for _, tail in entity_relations.get(q, []):
            if tail in local_set:
                edges.append([idx[q], idx[tail]])

    edge_index = (torch.tensor(edges, dtype=torch.long).t().contiguous()
                  if edges else torch.empty((2, 0), dtype=torch.long))

    return Data(x=x, edge_index=edge_index)

In [ ]:
from tqdm import tqdm

article_graphs = {}
skipped = 0
for aid, entities_dict in tqdm(article_entities.items()):
    g = build_article_graph(entities_dict)
    if g is not None:
        article_graphs[aid] = g
    else:
        skipped += 1

with open(GRAPH_PATH, 'wb') as f:
    pickle.dump(article_graphs, f)

print(f"Total graphs: {len(article_graphs)} | Skipped: {skipped}")

100%|██████████| 12529/12529 [00:01<00:00, 10823.35it/s]


Total graphs: 12331 | Skipped: 198


## Graphs representation

In [ ]:
import numpy as np

node_counts = [g.x.shape[0] for g in article_graphs.values()]
edge_counts = [g.edge_index.shape[1] for g in article_graphs.values()]
isolated = sum(1 for e in edge_counts if e == 0)

print(f"Nodes per graph  -> mean: {np.mean(node_counts):.1f}, median: {np.median(node_counts):.0f}, max: {max(node_counts)}")
print(f"Edges per graph  -> mean: {np.mean(edge_counts):.1f}, median: {np.median(edge_counts):.0f}, max: {max(edge_counts)}")
print(f"Graphs with zero internal edges (isolated nodes only): {isolated} ({100*isolated/len(article_graphs):.1f}%)")

# Peek at one real example
sample_id = list(article_graphs.keys())[0]
sample = article_graphs[sample_id]
print(f"\nSample article '{sample_id}': {sample.x.shape[0]} nodes, {sample.edge_index.shape[1]} edges")

Nodes per graph  -> mean: 5.4, median: 5, max: 22
Edges per graph  -> mean: 2.7, median: 2, max: 63
Graphs with zero internal edges (isolated nodes only): 4110 (33.3%)

Sample article 'train_0': 4 nodes, 0 edges


## GNN Module

##GAT

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool

class ArticleGNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=128, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim, heads=heads, concat=True)
        self.conv2 = GATConv(hidden_dim * heads, out_dim, heads=1, concat=False)

    def forward(self, x, edge_index, batch):
        x = F.elu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return global_mean_pool(x, batch)

In [ ]:
KG_DIM = entity_embeddings.shape[1]
gnn_test = ArticleGNN(in_dim=KG_DIM)

sample_id = list(article_graphs.keys())[0]
sample = article_graphs[sample_id]
batch_index = torch.zeros(sample.x.shape[0], dtype=torch.long)
out = gnn_test(sample.x, sample.edge_index, batch_index)
print(f"'{sample_id}': {sample.x.shape[0]} nodes, {sample.edge_index.shape[1]} edges -> embedding shape {out.shape}")

isolated_id = next(aid for aid, g in article_graphs.items() if g.edge_index.shape[1] == 0)
isolated = article_graphs[isolated_id]
batch_index2 = torch.zeros(isolated.x.shape[0], dtype=torch.long)
out2 = gnn_test(isolated.x, isolated.edge_index, batch_index2)
print(f"'{isolated_id}' (isolated): {isolated.x.shape[0]} nodes, 0 edges -> embedding shape {out2.shape}")

'train_0': 4 nodes, 0 edges -> embedding shape torch.Size([1, 128])
'train_0' (isolated): 4 nodes, 0 edges -> embedding shape torch.Size([1, 128])


In [ ]:
# ============================================================
# SECTION 3: DATASET WRAPPER
# FakeNet_dataset's __getitem__ can silently return a DIFFERENT
# row than requested (its internal retry loop re-randomizes the
# index on invalid images). So we match each returned item to its
# graph via the article's CONTENT TEXT (always correct, regardless
# of which row was actually returned) instead of by list position.
# ============================================================

# Build content -> article_id lookup from both train and test dataframes
content_to_article_id = {}
for i, row in train_df.iterrows():
    content_to_article_id[row['content']] = f'train_{i}'
for i, row in test_df.iterrows():
    content_to_article_id[row['content']] = f'test_{i}'

print(f"Content lookup built: {len(content_to_article_id)} entries")

Content lookup built: 12361 entries


In [ ]:
from data.FakeNet_dataset import FakeNet_dataset
from torch_geometric.data import Data

class FakeNetKGDataset(FakeNet_dataset):
    """
    Identical to the baseline FakeNet_dataset -- no logic changed --
    just adds the matching article graph to each returned item,
    looked up by content text (safe against the retry-index issue).
    """
    def __init__(self, article_graphs, content_to_article_id, kg_dim, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.article_graphs = article_graphs
        self.content_to_article_id = content_to_article_id
        self.kg_dim = kg_dim
        self._empty_graph = Data(x=torch.zeros((1, kg_dim)), edge_index=torch.empty((2, 0), dtype=torch.long))

    def __getitem__(self, index):
        (content, img_GT, img_GT2, label, category), GT_path = super().__getitem__(index)

        article_id = self.content_to_article_id.get(content)
        graph = self.article_graphs.get(article_id, self._empty_graph) if article_id else self._empty_graph

        return (content, img_GT, img_GT2, label, category), GT_path, graph

In [ ]:
# ------------------------------------------------------------
# 3.1 — Matching collate function (extends collate_fn_english
# from train_vimoe.py, adds graph batching)
# ------------------------------------------------------------

from torch_geometric.data import Batch
from transformers import BertTokenizer, CLIPProcessor

word_token_length = 197
token_uncased = BertTokenizer.from_pretrained("bert-base-uncased")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

def collate_fn_english_kg(data):
    sents = [i[0][0] for i in data]
    image = [i[0][1] for i in data]
    image_aug = [i[0][2] for i in data]
    labels = [i[0][3] for i in data]
    category = [0 for i in data]
    GT_path = [i[1] for i in data]
    graphs = [i[2] for i in data]

    token_data = token_uncased(
        text=sents, truncation=True, padding="max_length",
        max_length=word_token_length, return_tensors="pt", return_length=True,
    )
    clip_inputs = clip_processor(
        text=sents, images=image, truncation=True, padding="max_length",
        max_length=77, return_tensors="pt", return_length=True,
    )

    input_ids = token_data["input_ids"]
    attention_mask = token_data["attention_mask"]
    token_type_ids = token_data["token_type_ids"]
    image = torch.stack(image)
    image_aug = torch.stack(image_aug)
    labels = torch.LongTensor(labels)
    category = torch.LongTensor(category)
    graph_batch = Batch.from_data_list(graphs)

    return (
        (input_ids, attention_mask, token_type_ids),
        (image, image_aug, labels, category, sents),
        clip_inputs,
        GT_path,
        graph_batch,
    )

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
# ------------------------------------------------------------
# 3.2 — Build train/validate KG-augmented datasets + loaders
# (matches train_vimoe.py's own instantiation pattern exactly)
# ------------------------------------------------------------

GT_size = 224
custom_batch_size = 4

train_kg_dataset = FakeNetKGDataset(
    article_graphs=article_graphs,
    content_to_article_id=content_to_article_id,
    kg_dim=KG_DIM,
    is_train=True,
    dataset="gossip",
    image_size=GT_size,
    data_augment=False,
    duplicate_fake_times=0,
)

validate_kg_dataset = FakeNetKGDataset(
    article_graphs=article_graphs,
    content_to_article_id=content_to_article_id,
    kg_dim=KG_DIM,
    is_train=False,
    dataset="gossip",
    image_size=GT_size,
)

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_kg_dataset, batch_size=custom_batch_size, shuffle=True,
    collate_fn=collate_fn_english_kg, num_workers=4, drop_last=True, pin_memory=True,
)
validate_loader = DataLoader(
    validate_kg_dataset, batch_size=custom_batch_size, shuffle=False,
    collate_fn=collate_fn_english_kg, num_workers=4, drop_last=False, pin_memory=True,
)

print(f"Train dataset size: {len(train_kg_dataset)}")
print(f"Validate dataset size: {len(validate_kg_dataset)}")

sample_batch = next(iter(train_loader))
print(f"Batch elements: {len(sample_batch)}")  # should be 5: texts, others, clip_inputs, GT_path, graph_batch
print(f"Graph batch: {sample_batch[4]}")

duplicate_fake_times: 0
Dataset: gossip
Using More Negative Examples: False
We are resampling bad examples using randint
Workbook name /content/dataset/AAAI_dataset/gossip_train.xlsx


100%|██████████| 9759/9759 [00:00<00:00, 106911.42it/s]


real news: 7747
fake news: 2013
thresh: 0.79375


100%|██████████| 9759/9759 [00:00<00:00, 57794.68it/s]


Skipped Num 0
duplicate_fake_times: 0
Dataset: gossip
Using More Negative Examples: False
We are resampling bad examples using randint
Workbook name /content/dataset/AAAI_dataset/gossip_test.xlsx


...

real news: 2229
fake news: 542
thresh: 0.8044027426921689


100%|██████████| 2770/2770 [00:00<00:00, 60388.91it/s]

Skipped Num 0
Train dataset size: 9759
Validate dataset size: 2770



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Batch elements: 5
Graph batch: DataBatch(x=[23, 512], edge_index=[2, 37], batch=[23], ptr=[5])
